# Claude Code 2.1.314~2.1.316 바뀐 점과 쓰는 법: blockWritesOutsideWorkingDirectories로 작업 폴더 밖 수정 막기, CLAUDE_CODE_SUBAGENT_EFFORT·ultrathink 끄기 환경변수, Opus 4.8 fast mode 종료(11/30) 대비

위에서부터 차례로 실행해 보세요.

### 준비: API 키 넣기

Colab 왼쪽의 🔑(보안 비밀)에 `ANTHROPIC_API_KEY` 를 자기 값으로 추가하고 노트북 액세스를 켜 주세요. 키는 노트북에 저장되지 않아요.

In [ ]:
import os
from google.colab import userdata

for name in ['ANTHROPIC_API_KEY']:
    os.environ[name] = userdata.get(name)

### Claude Code 업데이트와 버전 확인

In [ ]:
%%bash
set -eo pipefail
npm i -g @anthropic-ai/claude-code@2.1.314 2>&1 | tail -3
claude --version
claude update 2>&1 | tail -5
claude --version
npm view @anthropic-ai/claude-code time --json | grep -E '"2\.1\.31[4-6]"'

### 프로젝트 폴더와 설정 만들기

In [ ]:
%%bash
mkdir -p /work/proj/.claude && cd /work/proj
echo '{"permissions":{"blockWritesOutsideWorkingDirectories":true}}' > .claude/settings.json
rm -f /tmp/outside.txt

### 차단 켬: Write로 /tmp 쓰기 시도

In [ ]:
%%bash
cd /work/proj; export CI=1
claude -p 'Use the Write tool to create /tmp/outside.txt with content hello. Report in one sentence.' --allowedTools Write --max-turns 5
ls -la /tmp/outside.txt || echo 'file not created'

### 차단 끔: 같은 요청

In [ ]:
%%bash
cd /work/proj; export CI=1
echo '{}' > .claude/settings.json
claude -p 'Use the Write tool to create /tmp/outside.txt with content hello. Report in one sentence.' --allowedTools Write --max-turns 5
ls -la /tmp/outside.txt; rm -f /tmp/outside.txt
echo '{"permissions":{"blockWritesOutsideWorkingDirectories":true}}' > .claude/settings.json

### 차단 켬 + Bash로 /tmp 쓰기

In [ ]:
%%bash
cd /work/proj; export CI=1
claude -p 'Use the Bash tool to run: echo hello > /tmp/outside.txt . Report in one sentence.' --allowedTools Bash --max-turns 5
ls -la /tmp/outside.txt && cat /tmp/outside.txt; rm -f /tmp/outside.txt

### effort 확인용 로컬 프록시

In [ ]:
%%bash
cat > /work/proxy.py <<'E'
import http.server, urllib.request, json, sys
LOG=sys.argv[1]
class H(http.server.BaseHTTPRequestHandler):
    def do_POST(self):
        n=int(self.headers.get('content-length',0)); body=self.rfile.read(n)
        try:
            j=json.loads(body)
            open(LOG,'a').write(json.dumps({'effort':(j.get('output_config') or {}).get('effort'),'n_tools':len(j.get('tools',[]))})+'\n')
        except Exception: pass
        hdr={k:v for k,v in self.headers.items() if k.lower() not in('host','accept-encoding','content-length')}
        req=urllib.request.Request('https://api.anthropic.com'+self.path,data=body,headers=hdr,method='POST')
        try: r=urllib.request.urlopen(req); code=r.status
        except urllib.error.HTTPError as e: r=e; code=e.code
        self.send_response(code)
        for k,v in r.getheaders():
            if k.lower() not in('transfer-encoding','connection','content-length'): self.send_header(k,v)
        self.send_header('Connection','close'); self.end_headers()
        while True:
            c=r.read(1024)
            if not c: break
            self.wfile.write(c); self.wfile.flush()
    def log_message(self,*a): pass
http.server.ThreadingHTTPServer(('127.0.0.1',8899),H).serve_forever()
E
rm -f /work/api.jsonl
nohup python3 /work/proxy.py /work/api.jsonl > /work/proxy.log 2>&1 &
sleep 1; echo proxy started

### SUBAGENT_EFFORT=low: 메인은 high, 서브에이전트는 low

In [ ]:
%%bash
cd /work/proj; export CI=1 ANTHROPIC_BASE_URL=http://127.0.0.1:8899
rm -f /work/api.jsonl
CLAUDE_CODE_SUBAGENT_EFFORT=low claude -p 'Use the Agent tool to spawn a subagent (general-purpose) that answers: what is 17*23? Then report its answer.' --effort high --max-turns 6 > /dev/null
cat /work/api.jsonl